# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


I chose Lane 2 Refresh / Content Opportunity Scoring.
ML task type: Ranking / Scoring
The goal is to rank pages based on how likely they are to need a content refresh, expansion, or monitoring. Instead of predicting Google's ranking algorithm, the model helps SEO teams prioritize which pages should be reviewed first. This is a ranking problem because the output is an ordered list of pages rather than a simple yes/no prediction.

**The action it supports:** An SEO strategist or account manager opens that list on Monday morning to triage the pages most at risk. For the top-ranked pages, they might rewrite the title tag to improve a low click-through rate or expand the content of a thin page to boost its search presence. Having this prioritized list allows them to focus limited content resources on protecting the most traffic at risk, instead of auditing pages one by one.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining_label` = 1 when a page's `trend_direction` is "down", otherwise 0.

Predicting "traffic actually went down" is a better target because it teaches the model to discover genuine historical traffic loss from complex, underlying patterns in engagement and content signals. If you predict a score you built yourself from CTR and position, the model isn't learning to find at-risk pages; it is just approximating your manual formula, making the machine learning step redundant.

If you gave `trend_direction` and `trend_pct` to the model as inputs, you would cause data leakage. Because the label is calculated directly from those columns, the model would find the exact answer hidden in the data, making your evaluation metrics look perfect while meaning absolutely nothing. Once deployed in real use on new data where the future trend is still unknown, the model would completely fall apart.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

I will evaluate the ranking using **Precision@50**.

Precision@50 measures how many of the top 50 pages recommended by the model are actually good candidates for refresh or review. A higher Precision@50 means the SEO team spends more time reviewing pages that are likely to benefit from action rather than pages that do not need attention.

**Cost of mistakes**

* False Positive (Healthy page in the top 50): The strategist wastes precious Monday morning time manually auditing a page that doesn't actually need a refresh.
* False Negative (Misses a declining page): The page continues to lose traffic silently, but the strategist remains unaware because it didn't make the list.
* What Precision@50 protects against: It protects against false positives. By optimizing the top 50 slots, it ensures that when the strategist opens the list on Monday morning, their limited time is not wasted on false alarms.
* What it ignores: misses outside the top 50. With about 16,000 declining pages and only 50 reviewed, most declining pages will never make the list anyway. I accept that trade-off because the strategist's time is the real limit.

**What counts as "good"**

If 54% of all pages are declining, picking 50 pages completely at random would yield an expected Precision@50 of roughly 54% (or 27 correct pages). This baseline tells us that 54% is the absolute floor our model has to beat; any model scoring near or below this baseline is performing no better than picking at random and adds zero algorithmic value.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
print("Unique content_id:", df["content_id"].nunique())
df.head()

Rows: 30000
Columns: 44
Unique content_id: 30000


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


Each row represents **one content page** with aggregated search and engagement metrics.
The model will make one prediction for each page, producing a ranked list of pages that should be reviewed first.

In [2]:
# Sketch of the target column (same definition as scripts/01_prepare_features.py)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(df["is_declining_label"].value_counts())
print("Base rate:", round(df["is_declining_label"].mean(), 3))
df[["content_id", "content_type", "impressions_90d", "ctr", "avg_position", "is_declining_label"]].head(10)

is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Base rate: 0.542


,content_id,content_type,impressions_90d,ctr,avg_position,is_declining_label
0,content_304f48230142,keyword article,3803,0.76,10.6,1
1,content_a1fb4e703a9e,keyword article,15320,0.05,20.3,1
2,content_9aa793d4d895,keyword article,12581,0.09,36.5,1
3,content_331d6c4de07b,keyword article,11751,0.49,6.2,0
4,content_d99b7a2d90ca,keyword article,19140,0.13,44.0,1
5,content_d4084a4bc775,keyword article,3970,0.03,8.5,1
6,content_9a34b442b552,keyword article,20,0.00,7.0,1
7,content_a63219c6e95a,keyword article,1724,0.06,21.2,0
8,content_5e6c160719bc,keyword article,32574,0.09,46.0,1
9,content_c27558df2b0c,keyword article,1240,0.16,4.9,1


The target column is already fairly balanced (54.2% declining), so the model is not chasing a rare event. The hard part is putting the *right* declining pages at the top.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Results (Precision@50):** random picking ≈ 0.54 · hand-written rules baseline = **0.24** · Random Forest (tested on clients it never saw in training) = **0.68**.

**Why a rule struggles**

A fixed rule struggles because it cannot handle context. For example, a low CTR of 1% is a major red flag for a page ranking at Position 3 (where expected CTR is high), meaning the title tag or snippet is failing to attract clicks. However, that exact same 1% CTR at Position 30 is actually incredibly high for a page sitting on page three of Google. A rigid rule flags both pages equally, failing to see that the second page may not need a content refresh at all.

**The surprising number**

The handwritten rules likely scored worse than random (0.24 vs. 0.54) because human intuition often creates "optimism biases" or hyper-specific rules that accidentally filter out the actual problem areas. For instance, a strategist might write a rule targeting "high impressions but low clicks," assuming these are the pages in decline. However, in reality, the pages dropping fastest might be older, core pages whose impressions and clicks are both steadily tanking together. By focusing strictly on a flawed human theory, the rule actively steered the strategist away from the actual 54% base rate of declining pages.

**An honest limit**

This is anonymized starter data: a single 90-day snapshot, and the label is a proxy for decline, not a confirmed business outcome. So I cannot claim "the model works" globally. SEO data is also highly dependent on the specific client, industry, and seasonality. Because this Random Forest was evaluated on clients it never saw in training, a 0.68 score is directional evidence that it generalizes across the 32 clients in this dataset. However, if a new client joins the agency from a hyper-niche industry, or if Google rolls out a massive core algorithm update that changes baseline user behavior, the model's learned patterns may completely break down until it is retrained on the new data landscape.

In [3]:
print("Pages:", len(df))
print("Average CTR (%):", round(df["ctr"].mean(), 4))
print("Average Position:", round(df[df["avg_position"] > 0]["avg_position"].mean(), 2))
print()
print("Content Types")
print(df["content_type"].value_counts())

Pages: 30000
Average CTR (%): 0.5107
Average Position: 17.03

Content Types
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.